# 📐 Statistics Cheatsheet — DSI SquarePoint

Полная шпаргалка по статистике для Dataset Interview.  
Для каждого метода: **когда применять**, **конкретный кейс из DSI**, **как читать результат**.

---

## Содержание

1. [Setup & синтетические данные](#1)
2. [Описательная статистика](#2)
3. [Тесты нормальности](#3)
4. [Сравнение двух групп](#4)
5. [Сравнение трёх и более групп](#5)
6. [Корреляция](#6)
7. [Категориальные vs категориальные](#7)
8. [Проверка гипотез — фреймворк](#8)
9. [Множественное тестирование](#9)
10. [Information Coefficient (IC/ICIR)](#10)
11. [🚀 Карта решений — какой тест выбрать](#11)
12. [🚀 Copy-paste шаблон для DSI](#12)


## 1. Setup & синтетические данные

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (
    shapiro, normaltest, kstest,          # нормальность
    ttest_ind, ttest_rel,                 # t-тесты
    mannwhitneyu, wilcoxon,              # непараметрические аналоги
    f_oneway, kruskal,                   # ANOVA и аналог
    pearsonr, spearmanr, kendalltau,     # корреляция
    chi2_contingency, fisher_exact,      # категориальные
    levene, bartlett,                    # равенство дисперсий
)
from statsmodels.stats.multitest import multipletests  # множественное тестирование
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
pd.set_option('display.float_format', '{:.4f}'.format)
SEED = 42
np.random.seed(SEED)

# ── Синтетические данные (имитируем типичный DSI датасет) ─────────────────
n = 500
df = pd.DataFrame({
    # Числовые биомаркеры / метрики
    'glucose':      np.random.normal(100, 20, n),
    'bmi':          np.random.normal(26, 5, n),
    'age':          np.random.randint(20, 80, n).astype(float),
    'revenue':      np.random.exponential(1000, n),   # скошенный
    'rating':       np.random.uniform(1, 5, n),

    # Категориальные
    'group':        np.random.choice(['control', 'treatment'], n),
    'segment':      np.random.choice(['A', 'B', 'C'], n, p=[0.5, 0.3, 0.2]),
    'converted':    np.random.binomial(1, 0.3, n),

    # Таргет
    'stage':        np.random.choice([0, 1, 2, 3], n, p=[0.4, 0.3, 0.2, 0.1]),
})

# Делаем treatment реально лучше (иначе неинтересно)
df.loc[df['group'] == 'treatment', 'glucose'] += 8
df.loc[df['group'] == 'treatment', 'converted'] = np.random.binomial(1, 0.42, (df['group']=='treatment').sum())

print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(df.head(3))

## 2. Описательная статистика

**Всегда первый шаг.** Прежде чем запускать тесты — понять что перед тобой.


### 2.1 Основные метрики и их интерпретация

| Метрика | Формула | Что говорит | Когда важно |
|---|---|---|---|
| **Mean** | Σx/n | Центр тяжести | Чувствителен к выбросам |
| **Median** | 50й перцентиль | Устойчивый центр | При скошенных данных |
| **Std** | √Var | Разброс | Всегда смотри рядом с mean |
| **Skewness** | E[(x-μ)³]/σ³ | Асимметрия | > 1 или < -1 → нелинейность |
| **Kurtosis** | E[(x-μ)⁴]/σ⁴ - 3 | Тяжесть хвостов | > 3 → выбросы влияют на RMSE |
| **CV** | std/mean | Относительный разброс | Сравнение разных шкал |


In [ ]:
def describe_extended(series, name=''):
    """Расширенная описательная статистика с интерпретацией."""
    s = series.dropna()
    skew = s.skew()
    kurt = s.kurt()
    cv   = s.std() / s.mean() if s.mean() != 0 else np.nan

    print(f'── {name or series.name} ──────────────────────────────')
    print(f'  n={len(s):,}  missing={series.isna().sum()}')
    print(f'  mean={s.mean():.3f}  median={s.median():.3f}  '
          f'std={s.std():.3f}  CV={cv:.3f}')
    print(f'  min={s.min():.3f}  Q1={s.quantile(.25):.3f}  '
          f'Q3={s.quantile(.75):.3f}  max={s.max():.3f}')
    print(f'  skew={skew:.3f}  ', end='')

    # Интерпретация skewness
    if skew > 1:    print('→ ПРАВЫЙ СКОС — log-transform, медиана > mean')
    elif skew < -1: print('→ ЛЕВЫЙ СКОС — ceiling effect, mean > медиана')
    else:           print('→ приблизительно симметрично')

    print(f'  kurt={kurt:.3f}  ', end='')
    if kurt > 3:  print('→ ТЯЖЁЛЫЕ ХВОСТЫ — выбросы влияют на RMSE')
    elif kurt < -1: print('→ плосковершинное распределение')
    else:           print('→ хвосты в норме')
    print()

for col in ['glucose', 'bmi', 'revenue', 'rating']:
    describe_extended(df[col])

## 3. Тесты нормальности

**Зачем:** от нормальности зависит выбор теста — параметрический (t-test) или непараметрический (Mann-Whitney).

**Практическое правило DSI:**
- n > 5000 → все тесты почти всегда отвергают H₀ даже при несущественных отклонениях → смотри на QQ plot визуально
- n < 50 → Shapiro-Wilk обязателен
- 50 < n < 5000 → оба подхода


### 3.1 Какой тест нормальности выбрать

| Тест | Лучший при | Слабость |
|---|---|---|
| **Shapiro-Wilk** | n < 5000, лучший по мощности | Медленный при больших n |
| **D'Agostino-Pearson** | n > 20, любой размер | Менее мощный чем Shapiro |
| **KS-test** | Проверка против любого распределения | Слабее для нормальности |
| **QQ plot** | Всегда — визуальная диагностика | Субъективный |


In [ ]:
def test_normality(series, name='', alpha=0.05):
    """
    Запускает три теста нормальности и выдаёт вердикт.

    H₀: данные нормально распределены
    H₁: данные не нормально распределены

    p > alpha → не отвергаем H₀ → данные совместимы с нормальным распределением
    p ≤ alpha → отвергаем H₀ → используй непараметрические тесты
    """
    s = series.dropna()
    print(f'── Normality tests: {name or series.name} (n={len(s):,}) ──')

    # 1. Shapiro-Wilk — лучший для n < 5000
    if len(s) <= 5000:
        stat, p = shapiro(s)
        verdict = '✓ normal' if p > alpha else '✗ NOT normal'
        print(f'  Shapiro-Wilk:       W={stat:.4f}  p={p:.4f}  → {verdict}')

    # 2. D'Agostino-Pearson — для любого n
    stat, p = normaltest(s)
    verdict = '✓ normal' if p > alpha else '✗ NOT normal'
    print(f"  D'Agostino-Pearson: stat={stat:.4f}  p={p:.4f}  → {verdict}")

    # 3. KS-test против стандартной нормали (данные нужно стандартизировать)
    s_std = (s - s.mean()) / s.std()
    stat, p = kstest(s_std, 'norm')
    verdict = '✓ normal' if p > alpha else '✗ NOT normal'
    print(f'  Kolmogorov-Smirnov: D={stat:.4f}  p={p:.4f}  → {verdict}')
    print()

test_normality(df['glucose'], 'glucose (should be ~normal)')
test_normality(df['revenue'], 'revenue (skewed, should NOT be normal)')

In [ ]:
# QQ Plot — визуальная проверка нормальности
# Если точки лежат на диагонали → нормальное распределение
# Загибы вверх на краях → тяжёлые хвосты (t-distribution)
# S-образный изгиб → скошенное распределение

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes,
                           ['glucose', 'revenue'],
                           ['glucose — ожидаем нормальное', 'revenue — скошенное']):
    (osm, osr), (slope, intercept, r) = stats.probplot(df[col], dist='norm')
    ax.plot(osm, osr, 'o', alpha=0.4, markersize=4, color='steelblue')
    ax.plot(osm, slope * np.array(osm) + intercept, 'r--', linewidth=1.5)
    ax.set_title(f'QQ Plot: {title}\nR={r:.3f}  (1.0 = идеально нормальное)')
    ax.set_xlabel('Theoretical quantiles')
    ax.set_ylabel('Sample quantiles')

sns.despine()
plt.tight_layout()
plt.show()

print('Интерпретация QQ Plot:')
print('  R > 0.99 → нормальное')
print('  R 0.97-0.99 → незначительное отклонение')
print('  R < 0.97 → используй непараметрические тесты')

## 4. Сравнение двух групп

**DSI кейс:** "Есть две группы ресторанов (дорогие / дешёвые). Отличается ли средний рейтинг?"  
**DSI кейс:** "Control vs Treatment — выше ли glucose в treatment группе?"

### Алгоритм выбора теста:

```
Данные нормальные И дисперсии равны?
    ├── ДА → t-test (равные дисперсии)
    ├── Нормальные, дисперсии НЕРАВНЫ → t-test Welch (equal_var=False)  ← default в scipy!
    └── НЕ нормальные → Mann-Whitney U
```


### 4.1 Проверка равенства дисперсий (Levene's test)

In [ ]:
# ВСЕГДА проверяй дисперсии перед t-test
# H₀: дисперсии равны
# H₁: дисперсии НЕ равны

ctrl = df.loc[df['group']=='control',   'glucose']
treat= df.loc[df['group']=='treatment', 'glucose']

# Levene — устойчив к ненормальности (рекомендуется)
stat, p = levene(ctrl, treat)
print(f'Levene test: stat={stat:.4f}  p={p:.4f}')
print(f'→ {"Дисперсии РАВНЫ → используй equal_var=True" if p > 0.05 else "Дисперсии НЕРАВНЫ → используй equal_var=False (Welch)"}')
print(f'  ctrl  std={ctrl.std():.3f}')
print(f'  treat std={treat.std():.3f}')

### 4.2 t-test (параметрический, нормальные данные)

In [ ]:
"""
t-test независимых выборок

Когда применять:
  - Данные нормально распределены (Shapiro p > 0.05 или n > 30 по ЦПТ)
  - Две независимые группы
  - Числовой таргет

DSI кейсы:
  - Control vs Treatment по конверсии/выручке
  - Мужчины vs женщины по HbA1c
  - До реформы vs после по доходности акций

H₀: средние двух групп равны (μ₁ = μ₂)
H₁: средние различаются (двусторонний) или μ₁ > μ₂ (односторонний)
"""

# equal_var=False → тест Welch (более безопасный default — не требует равных дисперсий)
# alternative: 'two-sided' | 'greater' | 'less'
stat, p = ttest_ind(treat, ctrl, equal_var=False, alternative='two-sided')

print('Independent t-test (Welch):')
print(f'  control   mean={ctrl.mean():.3f}  n={len(ctrl)}')
print(f'  treatment mean={treat.mean():.3f}  n={len(treat)}')
print(f'  t={stat:.4f}  p={p:.4f}')
print()

alpha = 0.05
if p < alpha:
    print(f'  p={p:.4f} < α={alpha} → Отвергаем H₀')
    print(f'  → Группы СТАТИСТИЧЕСКИ ЗНАЧИМО различаются')
    print(f'  → Разница средних: {treat.mean()-ctrl.mean():.3f}')
else:
    print(f'  p={p:.4f} ≥ α={alpha} → НЕ отвергаем H₀')
    print(f'  → Нет достаточных оснований утверждать различие')

# Effect size — Cohen's d
# < 0.2 маленький | 0.2-0.5 средний | 0.5-0.8 большой | > 0.8 очень большой
pooled_std = np.sqrt((ctrl.std()**2 + treat.std()**2) / 2)
cohens_d = (treat.mean() - ctrl.mean()) / pooled_std
print(f"\n  Effect size (Cohen's d) = {cohens_d:.3f}")
size = 'large' if abs(cohens_d) > 0.8 else 'medium' if abs(cohens_d) > 0.5 else 'small' if abs(cohens_d) > 0.2 else 'negligible'
print(f'  → {size} effect')
print('  Правило: p < 0.05 без effect size — мало смысла на большом n!')

### 4.3 Mann-Whitney U (непараметрический)

In [ ]:
"""
Mann-Whitney U test (Wilcoxon rank-sum test)

Когда применять:
  - Данные НЕ нормальные (Shapiro p < 0.05)
  - Ординальные данные (рейтинги 1-5)
  - Выбросы есть и удалять нельзя
  - Маленькие выборки (n < 30)

DSI кейсы:
  - Сравнение рейтингов ресторанов двух районов
  - Сравнение выручки (обычно скошена) двух сегментов
  - Glucose у диабетиков stage 1 vs stage 2

H₀: распределения двух групп одинаковы
H₁: одно распределение сдвинуто относительно другого
"""

# alternative: 'two-sided' | 'greater' | 'less'
# use_continuity=True — поправка на непрерывность (по умолчанию True)
stat, p = mannwhitneyu(treat, ctrl, alternative='two-sided')

print('Mann-Whitney U test:')
print(f'  control   median={ctrl.median():.3f}  n={len(ctrl)}')
print(f'  treatment median={treat.median():.3f}  n={len(treat)}')
print(f'  U={stat:.0f}  p={p:.4f}')

if p < 0.05:
    print(f'  → Распределения ЗНАЧИМО различаются (p<0.05)')
else:
    print(f'  → Различие незначимо (p≥0.05)')

# Effect size — rank-biserial correlation r
# |r| < 0.1 пренебрежимо | 0.1-0.3 малый | 0.3-0.5 средний | > 0.5 большой
r = 1 - (2 * stat) / (len(treat) * len(ctrl))
print(f'  Effect size (rank-biserial r) = {r:.3f}')
size = 'large' if abs(r) > 0.5 else 'medium' if abs(r) > 0.3 else 'small' if abs(r) > 0.1 else 'negligible'
print(f'  → {size} effect')

### 4.4 Парный t-test и Wilcoxon signed-rank (связанные выборки)

In [ ]:
"""
Парный t-test / Wilcoxon signed-rank

Когда применять:
  - ОДНИ И ТЕ ЖЕ объекты в двух условиях (до/после)
  - Пациент до лечения и после
  - Ресторан в будний день vs выходной

DSI кейс: 'Glucose одних и тех же пациентов до и после диеты снизился?'
"""

# Симулируем до/после для одних пациентов
np.random.seed(SEED)
before = np.random.normal(110, 15, 100)
after  = before - np.random.normal(5, 8, 100)  # реальное снижение ~5

# Парный t-test (нормальные разности)
stat_t, p_t = ttest_rel(before, after)
print('Paired t-test (до vs после):')
print(f'  before mean={before.mean():.3f}  after mean={after.mean():.3f}')
print(f'  t={stat_t:.4f}  p={p_t:.4f}')
print(f'  → {"Значимое снижение" if p_t < 0.05 else "Незначимо"}')

# Wilcoxon signed-rank — непараметрический аналог
stat_w, p_w = wilcoxon(before, after, alternative='two-sided')
print(f'\nWilcoxon signed-rank: stat={stat_w:.0f}  p={p_w:.4f}')
print(f'  → {"Значимое различие" if p_w < 0.05 else "Незначимо"}')

## 5. Сравнение трёх и более групп

**DSI кейс:** "Три сегмента клиентов (A, B, C) — есть ли разница в выручке?"  
**DSI кейс:** "Рейтинги в 4 районах города — одинаковые ли?"

**Почему нельзя просто сделать много t-тестов:** множественное тестирование надувает α. 3 группы → 3 сравнения → вероятность ошибки первого рода уже ~14% вместо 5%.


### 5.1 One-way ANOVA (параметрический)

In [ ]:
"""
One-way ANOVA — Analysis of Variance

Когда:
  - 3+ независимых групп
  - Данные нормальные
  - Дисперсии примерно равны (Levene p > 0.05)

H₀: все групповые средние равны (μ₁ = μ₂ = μ₃)
H₁: хотя бы одна группа отличается

ВАЖНО: ANOVA говорит 'хоть что-то различается',
но не ГДЕ. Для этого нужен post-hoc тест.
"""

groups = [df.loc[df['segment']==seg, 'glucose'].values for seg in ['A','B','C']]

# Проверяем предположения
print('Levene test (равенство дисперсий):')
stat_l, p_l = levene(*groups)
print(f'  stat={stat_l:.4f}  p={p_l:.4f}  → {"равны" if p_l > 0.05 else "НЕРАВНЫ"}')

# ANOVA
stat_f, p_f = f_oneway(*groups)
print(f'\nOne-way ANOVA:')
for seg, g in zip(['A','B','C'], groups):
    print(f'  {seg}: mean={g.mean():.3f}  std={g.std():.3f}  n={len(g)}')
print(f'  F={stat_f:.4f}  p={p_f:.4f}')

if p_f < 0.05:
    print('  → Значимые различия между группами (p<0.05)')
    print('  → Запускаем post-hoc тест чтобы найти ГДЕ')
else:
    print('  → Различия незначимы')

### 5.2 Post-hoc тест Tukey HSD

In [ ]:
"""
Tukey HSD (Honestly Significant Difference)

Когда: после значимого ANOVA — найти КАКИЕ пары различаются
Автоматически делает поправку на множественное тестирование.
"""

from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(
    endog=df['glucose'],       # числовая переменная
    groups=df['segment'],      # группировка
    alpha=0.05
)
print(tukey)
print()
print('Интерпретация:')
print('  reject=True  → пара значимо различается')
print('  reject=False → пара НЕ различается')
print('  meandiff     → разница средних (group2 - group1)')
print('  p-adj        → скорректированный p-value')

### 5.3 Kruskal-Wallis (непараметрический аналог ANOVA)

In [ ]:
"""
Kruskal-Wallis test

Когда:
  - 3+ групп
  - Данные НЕ нормальные или ординальные
  - Аналог ANOVA без требования нормальности

DSI кейс: 'Рейтинги (1-5) ресторанов в трёх районах — одинаковые?'
DSI кейс: 'Выручка (скошенная) трёх сегментов — есть ли разница?'
"""

stat_kw, p_kw = kruskal(*groups)
print('Kruskal-Wallis test:')
for seg, g in zip(['A','B','C'], groups):
    print(f'  {seg}: median={np.median(g):.3f}  n={len(g)}')
print(f'  H={stat_kw:.4f}  p={p_kw:.4f}')
print(f'  → {"Значимые различия" if p_kw < 0.05 else "Различия незначимы"}')
print()
print('Post-hoc для Kruskal-Wallis: попарные Mann-Whitney с поправкой Bonferroni')

# Попарные Mann-Whitney с поправкой Bonferroni
from itertools import combinations
segs  = ['A', 'B', 'C']
pairs = list(combinations(segs, 2))
p_vals = []
for s1, s2 in pairs:
    g1 = df.loc[df['segment']==s1, 'glucose']
    g2 = df.loc[df['segment']==s2, 'glucose']
    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
    p_vals.append(p)

# Bonferroni поправка
reject, p_adj, _, _ = multipletests(p_vals, method='bonferroni')
print('\nПопарные Mann-Whitney (Bonferroni corrected):')
for (s1,s2), p_raw, p_a, rej in zip(pairs, p_vals, p_adj, reject):
    print(f'  {s1} vs {s2}: p_raw={p_raw:.4f}  p_adj={p_a:.4f}  reject={rej}')

## 6. Корреляция

**DSI кейс:** "Связан ли BMI с glucose? Линейно или нет?"  
**DSI кейс:** "Какие признаки сильнее всего коррелируют с таргетом?"


### 6.1 Какую корреляцию выбрать

| Метрика | Когда | Что измеряет | Выброс-устойчивость |
|---|---|---|---|
| **Pearson** | Оба нормальные, линейная связь | Линейная корреляция | Нет — чувствителен |
| **Spearman** | Любые, монотонная связь | Ранговая корреляция | Да — устойчив |
| **Kendall τ** | Маленькие n, много связанных рангов | Ранговая корреляция | Да, сильнее Spearman |


In [ ]:
def full_correlation(x, y, name_x='X', name_y='Y'):
    """
    Считает все три вида корреляции с интерпретацией.

    Интерпретация |r| / |ρ|:
      0.0 - 0.1  → пренебрежимая
      0.1 - 0.3  → слабая
      0.3 - 0.5  → умеренная
      0.5 - 0.7  → сильная
      0.7 - 1.0  → очень сильная

    Divergence (|Spearman| - |Pearson|):
      > 0.05 → нелинейная связь → деревья предпочтительнее Ridge
    """
    valid = pd.DataFrame({name_x: x, name_y: y}).dropna()
    x_, y_ = valid[name_x], valid[name_y]

    r_p, p_p = pearsonr(x_, y_)
    r_s, p_s = spearmanr(x_, y_)
    r_k, p_k = kendalltau(x_, y_)
    divergence = abs(r_s) - abs(r_p)

    print(f'── Correlation: {name_x} × {name_y} ──')
    print(f'  Pearson  r={r_p:+.4f}  p={p_p:.4f}  {"***" if p_p<0.001 else "**" if p_p<0.01 else "*" if p_p<0.05 else "n.s."}')
    print(f'  Spearman ρ={r_s:+.4f}  p={p_s:.4f}  {"***" if p_s<0.001 else "**" if p_s<0.01 else "*" if p_s<0.05 else "n.s."}')
    print(f'  Kendall  τ={r_k:+.4f}  p={p_k:.4f}')
    print(f'  Divergence = {divergence:+.4f}  → {"нелинейная связь → деревья" if divergence > 0.05 else "линейная связь → Ridge OK"}')

    # Сила связи по Spearman
    abs_s = abs(r_s)
    strength = 'очень сильная' if abs_s > 0.7 else 'сильная' if abs_s > 0.5 else \
               'умеренная' if abs_s > 0.3 else 'слабая' if abs_s > 0.1 else 'пренебрежимая'
    print(f'  Сила связи: {strength}  ({"положительная" if r_s > 0 else "отрицательная"})')
    print()

full_correlation(df['bmi'], df['glucose'], 'BMI', 'glucose')
full_correlation(df['age'], df['stage'],   'age', 'stage')

## 7. Категориальные vs категориальные

**DSI кейс:** "Зависит ли конверсия (да/нет) от сегмента клиента (A/B/C)?"  
**DSI кейс:** "Есть ли связь между полом пациента и стадией диабета?"


### 7.1 Chi-squared vs Fisher's Exact

| Тест | Когда |
|---|---|
| **Chi-squared** | Все ожидаемые частоты ≥ 5, любой размер таблицы |
| **Fisher's Exact** | Маленькие выборки, есть ячейки с ожидаемой частотой < 5, только 2×2 |


In [ ]:
"""
Chi-squared test of independence

H₀: переменные независимы
H₁: переменные зависимы (есть ассоциация)

DSI кейс: 'Зависит ли конверсия от сегмента?'
"""

# Таблица сопряжённости
contingency = pd.crosstab(df['segment'], df['converted'])
print('Contingency table (segment × converted):')
print(contingency)
print()

# chi2_contingency возвращает:
# chi2  — статистика
# p     — p-value
# dof   — степени свободы
# expected — ожидаемые частоты (все должны быть >= 5!)
chi2, p, dof, expected = chi2_contingency(contingency)

print(f'Chi-squared test:')
print(f'  χ²={chi2:.4f}  p={p:.4f}  dof={dof}')
print(f'  Min expected frequency: {expected.min():.2f}')
if expected.min() < 5:
    print('  ⚠ Ожидаемые частоты < 5 → используй Fisher exact (только для 2×2)')

if p < 0.05:
    print(f'  → Значимая ассоциация между сегментом и конверсией (p<0.05)')
else:
    print(f'  → Ассоциация незначима')

# Effect size — Cramér's V
# 0-0.1 слабый | 0.1-0.3 умеренный | 0.3-0.5 сильный | > 0.5 очень сильный
n = contingency.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
strength = 'strong' if cramers_v > 0.3 else 'moderate' if cramers_v > 0.1 else 'weak'
print(f"\n  Effect size (Cramér's V) = {cramers_v:.3f} → {strength} association")

In [ ]:
"""
Fisher's Exact test — для маленьких выборок или таблиц 2×2

DSI кейс: 'Связано ли наличие family_history с диагнозом диабета?'
"""

# Fisher работает только с таблицами 2×2
ct_2x2 = pd.crosstab(
    df['group'],
    df['converted']
)
print('2×2 table (group × converted):')
print(ct_2x2)

# alternative: 'two-sided' | 'greater' | 'less'
odds_ratio, p = fisher_exact(ct_2x2, alternative='two-sided')
print(f'\nFisher Exact test:')
print(f'  Odds Ratio = {odds_ratio:.3f}  p={p:.4f}')
print()

# Интерпретация Odds Ratio:
print('Интерпретация Odds Ratio (OR):')
print('  OR = 1.0  → нет ассоциации')
print('  OR > 1.0  → группа 1 имеет ВЫШЕ шанс события')
print('  OR < 1.0  → группа 1 имеет НИЖЕ шанс события')
print(f'  Твой OR={odds_ratio:.3f} → ', end='')
if abs(odds_ratio - 1) < 0.1:
    print('практически нет ассоциации')
elif odds_ratio > 1:
    print(f'treatment имеет в {odds_ratio:.1f}x выше шанс конверсии')
else:
    print(f'treatment имеет в {1/odds_ratio:.1f}x ниже шанс конверсии')

## 8. Фреймворк проверки гипотез

На DSI от тебя ожидают не просто "запустил тест", а **полный нарратив**.


### Структура правильного ответа:

```
1. НАБЛЮДЕНИЕ   — что видишь в данных
2. ГИПОТЕЗЫ     — H₀ и H₁ словами
3. ПРЕДПОЛОЖЕНИЯ — нормальность? независимость? размер выборки?
4. ТЕСТ         — какой и почему
5. РЕЗУЛЬТАТ    — статистика и p-value
6. ВЫВОД        — на языке задачи, не "p < 0.05"
7. EFFECT SIZE  — насколько практически значимо
```


In [ ]:
def hypothesis_test_full(group1, group2, name1='Group 1', name2='Group 2',
                          target_name='value', alpha=0.05):
    """
    Полный автоматический фреймворк:
    1. Проверка нормальности
    2. Проверка равенства дисперсий
    3. Выбор теста
    4. Запуск и интерпретация
    5. Effect size
    """
    g1, g2 = group1.dropna(), group2.dropna()

    print(f'══ Hypothesis Test: {name1} vs {name2} on {target_name} ══')
    print(f'  H₀: {target_name} одинаков в обеих группах')
    print(f'  H₁: {target_name} различается между группами')
    print(f'  α = {alpha}')
    print()

    # Step 1: нормальность
    normal1 = shapiro(g1[:5000])[1] > alpha if len(g1) <= 5000 else len(g1) > 30
    normal2 = shapiro(g2[:5000])[1] > alpha if len(g2) <= 5000 else len(g2) > 30
    both_normal = normal1 and normal2
    print(f'  Нормальность: {name1}={normal1}, {name2}={normal2}')

    # Step 2: равенство дисперсий (только если нормальные)
    if both_normal:
        _, p_lev = levene(g1, g2)
        equal_var = p_lev > alpha
        print(f'  Levene p={p_lev:.4f} → дисперсии {"равны" if equal_var else "неравны"}')
    else:
        equal_var = False

    # Step 3: выбор теста
    if both_normal:
        test_name = f't-test ({"equal var" if equal_var else "Welch"})'
        stat, p = ttest_ind(g1, g2, equal_var=equal_var)
    else:
        test_name = 'Mann-Whitney U'
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')

    print(f'  Тест: {test_name}')
    print()

    # Step 4: результат
    print(f'  {name1}: mean={g1.mean():.3f}  median={g1.median():.3f}  n={len(g1)}')
    print(f'  {name2}: mean={g2.mean():.3f}  median={g2.median():.3f}  n={len(g2)}')
    print(f'  stat={stat:.4f}  p={p:.4f}  {"***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "n.s."}')
    print()

    # Step 5: вывод
    if p < alpha:
        print(f'  ВЫВОД: Отвергаем H₀ (p={p:.4f} < α={alpha})')
        print(f'  → {target_name} ЗНАЧИМО различается между {name1} и {name2}')
        diff = g2.mean() - g1.mean()
        print(f'  → Разница средних: {diff:+.3f} ({diff/g1.mean()*100:+.1f}%)')
    else:
        print(f'  ВЫВОД: Не отвергаем H₀ (p={p:.4f} ≥ α={alpha})')
        print(f'  → Нет доказательств различия {target_name} между группами')

    # Step 6: effect size
    pooled_std = np.sqrt((g1.std()**2 + g2.std()**2) / 2)
    d = abs(g2.mean() - g1.mean()) / pooled_std if pooled_std > 0 else 0
    size = 'large' if d > 0.8 else 'medium' if d > 0.5 else 'small' if d > 0.2 else 'negligible'
    print(f"  Effect size (Cohen's d) = {d:.3f} → {size}")
    print()

# Демонстрация
hypothesis_test_full(
    df.loc[df['group']=='control',   'glucose'],
    df.loc[df['group']=='treatment', 'glucose'],
    name1='Control', name2='Treatment', target_name='glucose'
)

## 9. Множественное тестирование

**DSI кейс:** "Проверил 20 признаков на корреляцию с таргетом — нашёл 3 значимых. Можно верить?"

**Проблема:** при α=0.05 и 20 тестах ожидаешь ~1 ложно-значимый результат случайно.  
Вероятность хотя бы одной ошибки I рода = 1 − 0.95²⁰ ≈ **64%**.


### Поправки на множественное тестирование

| Метод | Когда | Строгость |
|---|---|---|
| **Bonferroni** | Мало тестов, хочешь максимальную строгость | Очень строгий |
| **Holm-Bonferroni** | Чуть менее строгий, чем Bonferroni | Строгий |
| **Benjamini-Hochberg (FDR)** | Много тестов (десятки/сотни признаков) | Умеренный — рекомендуется |
| **Без поправки** | 1-2 заранее сформулированных теста | — |


In [ ]:
# Симулируем тестирование 20 признаков на корреляцию с таргетом
np.random.seed(SEED)
n_features = 20
feature_matrix = np.random.randn(len(df), n_features)

# Несколько признаков реально связаны с таргетом
feature_matrix[:, 0] += df['stage'].values * 0.5  # реальная связь
feature_matrix[:, 1] += df['stage'].values * 0.3  # реальная связь

# Сырые p-values
raw_pvals = []
for i in range(n_features):
    _, p = spearmanr(feature_matrix[:, i], df['stage'])
    raw_pvals.append(p)

raw_pvals = np.array(raw_pvals)

print(f'Сырые результаты: {(raw_pvals < 0.05).sum()} из {n_features} значимы без поправки')
print()

# Применяем поправки
results = pd.DataFrame({'feature': [f'feat_{i}' for i in range(n_features)],
                         'p_raw': raw_pvals})

for method in ['bonferroni', 'holm', 'fdr_bh']:
    reject, p_adj, _, _ = multipletests(raw_pvals, method=method)
    results[f'p_{method}'] = p_adj
    results[f'sig_{method}'] = reject
    n_sig = reject.sum()
    print(f'{method:15s}: {n_sig} значимых после поправки')

print()
print('Итог: чем строже поправка, тем меньше значимых результатов.')
print('На DSI: при >10 тестах всегда применяй FDR (fdr_bh) и явно упоминай это.')
print()
print(results[['feature', 'p_raw', 'p_bonferroni', 'p_fdr_bh',
               'sig_bonferroni', 'sig_fdr_bh']].head(10).to_string(index=False))

## 10. Information Coefficient (IC / ICIR)

**Это специфика SquarePoint.** IC — главная метрика качества факторной модели в quant finance.

**IC** = Spearman ρ между предсказанными рангами и фактическими доходностями за период  
**ICIR** = IC_mean / IC_std — аналог Sharpe ratio для факторов


In [ ]:
"""
IC/ICIR — Information Coefficient / Information Coefficient Information Ratio

Контекст: у тебя есть фактор (predicted_return) и реальные доходности (actual_return)
по N акциям за T периодов.

IC за период t = Spearman(predicted_rank_t, actual_return_t)
ICIR = mean(IC) / std(IC)

Интерпретация IC:
  |IC| > 0.05   → слабый сигнал, но уже полезный
  |IC| > 0.10   → хороший сигнал
  |IC| > 0.15   → очень сильный (редкость на реальных данных)

Интерпретация ICIR:
  ICIR > 0.5    → стабильный сигнал
  ICIR > 1.0    → сильный и стабильный → фактор торгуем
  ICIR < 0.5    → нестабильно → нужна доработка или отбросить
"""

# Симулируем: 200 акций × 24 месяца
np.random.seed(SEED)
n_stocks, n_periods = 200, 24

# Предсказанные сигналы (наш 'фактор')
signals = np.random.randn(n_periods, n_stocks)

# Реальные доходности: частично коррелируют с сигналом (IC ≈ 0.08)
true_ic = 0.08
returns = true_ic * signals + np.sqrt(1 - true_ic**2) * np.random.randn(n_periods, n_stocks)

# Считаем IC за каждый период
ic_series = np.array([
    spearmanr(signals[t], returns[t]).statistic
    for t in range(n_periods)
])

ic_mean = ic_series.mean()
ic_std  = ic_series.std()
icir    = ic_mean / ic_std if ic_std > 0 else 0

# t-test: значимо ли IC отличается от 0?
t_stat, p_ic = stats.ttest_1samp(ic_series, popmean=0)

print('══ IC / ICIR Analysis ══')
print(f'  Periods       : {n_periods}')
print(f'  Stocks        : {n_stocks}')
print(f'  IC mean       : {ic_mean:.4f}  {"✓ сигнал есть" if abs(ic_mean) > 0.05 else "⚠ слабый сигнал"}')
print(f'  IC std        : {ic_std:.4f}')
print(f'  ICIR          : {icir:.4f}  {"✓ стабильный" if abs(icir) > 0.5 else "⚠ нестабильно"}')
print(f'  t-test vs 0   : t={t_stat:.4f}  p={p_ic:.4f}')
print(f'  Значим?       : {"ДА" if p_ic < 0.05 else "НЕТ"}')
print()

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.lineplot(x=range(n_periods), y=ic_series, color='steelblue',
             marker='o', markersize=4, ax=axes[0])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axhline(ic_mean, color='red', linestyle='--', linewidth=1.5,
                label=f'Mean IC={ic_mean:.3f}')
axes[0].fill_between(range(n_periods), 0, ic_series,
                      where=np.array(ic_series)>0, alpha=0.2, color='steelblue')
axes[0].fill_between(range(n_periods), 0, ic_series,
                      where=np.array(ic_series)<0, alpha=0.2, color='coral')
axes[0].set_title(f'IC по периодам  ICIR={icir:.2f}')
axes[0].set_xlabel('Period'); axes[0].set_ylabel('IC (Spearman ρ)')
axes[0].legend()

sns.histplot(ic_series, bins=15, kde=True, color='steelblue',
             edgecolor='white', ax=axes[1])
axes[1].axvline(0,       color='black', linewidth=1)
axes[1].axvline(ic_mean, color='red',   linestyle='--', linewidth=1.5,
                label=f'mean={ic_mean:.3f}')
axes[1].set_title('Распределение IC')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

## 11. 🚀 Карта решений — какой тест выбрать

```
ЧТО СРАВНИВАЕШЬ?
│
├── ЧИСЛОВУЮ переменную между группами
│   │
│   ├── 2 группы
│   │   ├── Данные НОРМАЛЬНЫЕ?
│   │   │   ├── ДА + дисперсии равны   → t-test (equal_var=True)
│   │   │   ├── ДА + дисперсии неравны → t-test Welch (equal_var=False) ← БЕЗОПАСНЫЙ DEFAULT
│   │   │   └── НЕТ / ординальные      → Mann-Whitney U
│   │   │
│   │   └── Те же объекты (до/после)?
│   │       ├── Нормальные разности → Paired t-test
│   │       └── НЕ нормальные       → Wilcoxon signed-rank
│   │
│   └── 3+ группы
│       ├── Нормальные + равные дисперсии → One-way ANOVA → Tukey post-hoc
│       └── НЕ нормальные / ординальные   → Kruskal-Wallis → Mann-Whitney + Bonferroni
│
├── СВЯЗЬ двух числовых переменных
│   ├── Оба нормальные, линейная связь → Pearson r
│   └── Хоть одна не нормальная / ординальная → Spearman ρ
│   └── Всегда считай оба + divergence
│
├── КАТЕГОРИАЛЬНУЮ vs КАТЕГОРИАЛЬНУЮ
│   ├── Все ожидаемые частоты >= 5 → Chi-squared
│   └── Есть ожидаемые < 5 (2×2)  → Fisher Exact
│
└── МНОГО ТЕСТОВ (> 5)?
    ├── Строго (мало тестов)     → Bonferroni / Holm
    └── Exploratory (много)      → Benjamini-Hochberg FDR
```

---

### Таблица p-value: что значит

| p-value | Обозначение | Вывод |
|---|---|---|
| p < 0.001 | *** | Очень высокая значимость |
| 0.001 ≤ p < 0.01 | ** | Высокая значимость |
| 0.01 ≤ p < 0.05 | * | Значимо (стандартный порог) |
| 0.05 ≤ p < 0.10 | . | Тренд, не значимо |
| p ≥ 0.10 | n.s. | Не значимо |

### Effect size: что значит

| Метрика | Малый | Средний | Большой |
|---|---|---|---|
| Cohen's d | 0.2 | 0.5 | 0.8 |
| Spearman ρ | 0.1 | 0.3 | 0.5 |
| Cramér's V | 0.1 | 0.3 | 0.5 |
| Rank-biserial r | 0.1 | 0.3 | 0.5 |

**Правило:** статистическая значимость ≠ практическая значимость. При n=100 000 даже разница в 0.001 будет значима. Всегда сообщай effect size.


## 12. 🚀 Copy-Paste шаблон для DSI

Вставь в свой ноутбук после EDA. Замени `TARGET`, `GROUP_COL`, `DATA_PATH`.


### Блок A — Описательная статистика + нормальность

In [ ]:
# ═══════════════════════════════════════════════════════
# БЛОК A: Описательная статистика + нормальность
# ═══════════════════════════════════════════════════════
TARGET    = 'stage'       # ← ЗАМЕНИТЬ
GROUP_COL = 'group'       # ← ЗАМЕНИТЬ (категориальная для сравнения)

num_cols = df.select_dtypes(include=np.number).columns.tolist()

# Расширенная описательная статистика
desc = df[num_cols].agg(['mean','median','std','skew','kurt']).T
desc['cv'] = desc['std'] / desc['mean'].abs()
desc['skew_flag'] = desc['skew'].apply(
    lambda s: '⚠ right-skew' if s > 1 else '⚠ left-skew' if s < -1 else 'OK'
)
print('Extended Descriptive Statistics:')
print(desc[['mean','median','std','cv','skew','skew_flag','kurt']].to_string())
print()

# Нормальность таргета
s = df[TARGET].dropna()
if len(s) <= 5000:
    stat_sw, p_sw = shapiro(s)
    print(f'Shapiro-Wilk [{TARGET}]: W={stat_sw:.4f}  p={p_sw:.4f}  '
          + ('→ нормальное' if p_sw > 0.05 else '→ НЕ нормальное → используй непараметрику'))
stat_da, p_da = normaltest(s)
print(f"D'Agostino [{TARGET}]: stat={stat_da:.4f}  p={p_da:.4f}")

### Блок B — Сравнение двух групп (авто-выбор теста)

In [ ]:
# ═══════════════════════════════════════════════════════
# БЛОК B: Сравнение двух групп (авто-выбор теста)
# ═══════════════════════════════════════════════════════
def auto_two_group_test(data, target, group_col, alpha=0.05):
    groups = data[group_col].dropna().unique()
    if len(groups) != 2:
        print(f'Ожидается 2 группы, найдено {len(groups)}: {groups}'); return

    g1 = data.loc[data[group_col]==groups[0], target].dropna()
    g2 = data.loc[data[group_col]==groups[1], target].dropna()

    print(f'Comparing {target} by {group_col}: {groups[0]} vs {groups[1]}')
    print(f'  {groups[0]}: n={len(g1)}  mean={g1.mean():.3f}  median={g1.median():.3f}')
    print(f'  {groups[1]}: n={len(g2)}  mean={g2.mean():.3f}  median={g2.median():.3f}')

    # Нормальность
    n1 = len(g1) > 30 or (shapiro(g1[:5000])[1] > alpha if len(g1) <= 5000 else True)
    n2 = len(g2) > 30 or (shapiro(g2[:5000])[1] > alpha if len(g2) <= 5000 else True)

    if n1 and n2:
        _, p_lev = levene(g1, g2)
        eq_var = p_lev > alpha
        stat, p = ttest_ind(g1, g2, equal_var=eq_var)
        test = f't-test (Welch={not eq_var})'
    else:
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
        test = 'Mann-Whitney U'

    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    print(f'  Test: {test}  stat={stat:.4f}  p={p:.4f}  {sig}')

    pooled_std = np.sqrt((g1.std()**2 + g2.std()**2) / 2)
    d = abs(g2.mean()-g1.mean()) / pooled_std if pooled_std > 0 else 0
    size = 'large' if d>0.8 else 'medium' if d>0.5 else 'small' if d>0.2 else 'negligible'
    print(f"  Cohen's d={d:.3f} → {size} effect")
    print(f'  → {"ЗНАЧИМО" if p < alpha else "не значимо"}')
    print()

auto_two_group_test(df, TARGET, GROUP_COL)

### Блок C — Корреляция всех признаков с таргетом

In [ ]:
# ═══════════════════════════════════════════════════════
# БЛОК C: Корреляция всех признаков с таргетом
# ═══════════════════════════════════════════════════════
feat_cols = [c for c in num_cols if c != TARGET]
rows = []
for c in feat_cols:
    valid = df[[c, TARGET]].dropna()
    r_p, p_p = pearsonr(valid[c], valid[TARGET])
    r_s, p_s = spearmanr(valid[c], valid[TARGET])
    rows.append({
        'feature':     c,
        'pearson':     r_p,   'p_pearson':  p_p,
        'spearman':    r_s,   'p_spearman': p_s,
        'abs_spearman': abs(r_s),
        'divergence':  abs(r_s) - abs(r_p),
        'sig': '***' if p_s<0.001 else '**' if p_s<0.01 else '*' if p_s<0.05 else 'n.s.'
    })

corr_df = pd.DataFrame(rows).sort_values('abs_spearman', ascending=False)

# FDR поправка на множественное тестирование
_, p_fdr, _, _ = multipletests(corr_df['p_spearman'].values, method='fdr_bh')
corr_df['p_fdr_bh'] = p_fdr
corr_df['sig_fdr']  = corr_df['p_fdr_bh'].apply(
    lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
)

print('Feature correlations with target (FDR corrected):')
print(corr_df[['feature','pearson','spearman','divergence','sig','sig_fdr']].to_string(index=False))
print()
print('divergence > 0.05 → нелинейная связь → деревья предпочтительнее')
print('sig_fdr = n.s. → значимость исчезла после поправки → может быть шум')

### Блок D — IC/ICIR (если данные с временной структурой)

In [ ]:
# ═══════════════════════════════════════════════════════
# БЛОК D: IC/ICIR (для данных с периодами)
# ═══════════════════════════════════════════════════════
def compute_ic(signals_df, returns_df):
    """
    signals_df : DataFrame (periods × assets) — предсказанные значения
    returns_df : DataFrame (periods × assets) — фактические доходности
    """
    ic_series = [
        spearmanr(signals_df.iloc[t], returns_df.iloc[t]).statistic
        for t in range(len(signals_df))
    ]
    ic_series = np.array(ic_series)
    ic_mean, ic_std = ic_series.mean(), ic_series.std()
    icir = ic_mean / ic_std if ic_std > 0 else 0
    _, p_val = stats.ttest_1samp(ic_series, popmean=0)

    print('═══ IC / ICIR ═══')
    print(f'  IC mean : {ic_mean:.4f}  {"✓" if abs(ic_mean)>0.05 else "⚠"}')
    print(f'  IC std  : {ic_std:.4f}')
    print(f'  ICIR    : {icir:.4f}  {"✓ торгуем" if abs(icir)>1.0 else "⚠ нестабильно" if abs(icir)>0.5 else "✗ слабо"}')
    print(f'  t-test  : p={p_val:.4f}  {"*** значим" if p_val<0.001 else "** значим" if p_val<0.01 else "* значим" if p_val<0.05 else "n.s."}')
    return ic_series

# ic = compute_ic(your_signals_df, your_returns_df)
print('Блок D: раскомментируй и подставь свои данные когда есть временная структура.')

---

## Финальная шпаргалка — что говорить на DSI

**Когда тебя спрашивают "значимы ли различия?":**
1. Формулируй H₀ и H₁ явно
2. Называй тест и почему его выбрал
3. p-value + effect size — оба обязательны
4. Вывод на языке задачи, не "p < 0.05"

**Красные флаги которые ищет SQ:**
- Тест на нормальность до t-test? → Если нет, -1 балл
- Равенство дисперсий проверил? → Если нет, -1 балл
- Множественное тестирование поправил? → Если нет, -1 балл
- Effect size посчитал? → Если нет, -0.5 балла
- IC/ICIR знаешь? → Если да, +2 балла

**Самая частая ошибка на DSI:**  
"p = 0.03, значит различие есть" на n = 50 000 — технически верно, но на практике разница может быть 0.001 и не иметь никакого экономического смысла. Всегда считай Cohen's d.
